# Private hosted agent to Teams without APIM

Provision a private Foundry environment, deploy the existing `demo-hosted-agent`, and publish it to Microsoft 365 / Teams by following [Publish agents to Microsoft 365 and Teams by using the REST API](https://learn.microsoft.com/azure/foundry/agents/how-to/publish-copilot-virtual-network), using the native `azure-ai-projects` SDK. Run the notebook section by section, not with Run All. Several cells create billable resources.

## Architecture

`Teams -> Azure Bot Service -> Foundry's source-IP-filtered public Activity route -> hosted agent -> model`

The Foundry account keeps **public network access disabled**. Teams traffic does **not** use Private Link: `enable_m365_public_endpoint` opens only the Activity Protocol route to Azure Bot Service / Microsoft 365 source IPs. Channel tokens, tenant checks, and RBAC still apply. Management and Responses calls stay private. Teams processes and stores messages under Microsoft 365 terms.

Infrastructure is [sample 11](https://github.com/microsoft-foundry/foundry-samples/tree/main/infrastructure/infrastructure-setup-bicep/11-private-network-basic-vnet) (VNet injection, private endpoint and DNS, model, private telemetry), pinned to a commit. ACR is disabled because the agent uses source-code deployment (`codeConfiguration`).

## Before starting

- Install `requirements.txt` into the kernel. You also need Git, Azure CLI with Bicep, azd >= 1.27.1, and the `azure.ai.agents` extension. Run `az login` and `azd auth login` in a terminal.
- Use a dedicated resource group, a region/model with quota, and non-overlapping address ranges. You need Owner (or Contributor + User Access Administrator) on the resource group.
- After provisioning, the kernel needs private connectivity: a VPN/ExpressRoute workstation with private DNS, or a VM in a connected **non-agent** subnet. The template doesn't create one. Portal publishing isn't supported when public network access is disabled.
- Use a Teams-enabled member account in the same tenant (no guests).


## 1. Configuration and clients

Settings come from the `PRIVATE_DEMO_*` section of the root `.env` (see [.env.example](.env.example)). The notebook reads that file into a local dictionary. It doesn't change `FOUNDRY_PROJECT_ENDPOINT` or other public settings used by notebooks 1 and 2.

The Bicep template adds a suffix to the account and project base names. The actual private endpoint is read from the deployment outputs, not from `.env`. The isolated azd environment in `private-agent/.azure/` gets the standard azd variable names that the [companion manifest](private-agent/azure.yaml) needs.


In [1]:
import json
import subprocess
from pathlib import Path
from uuid import uuid4

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    ActivityProtocolConfiguration, AgentEndpointConfig, BotServiceRbacAuthorizationScheme,
    EntraAuthorizationScheme, ProtocolConfiguration, ResponsesProtocolConfiguration,
)
from azure.core.exceptions import HttpResponseError
from azure.identity import AzureCliCredential
from azure.mgmt.authorization import AuthorizationManagementClient
from azure.mgmt.authorization.models import RoleAssignmentCreateParameters
from azure.mgmt.botservice import AzureBotService
from azure.mgmt.botservice.models import Bot, BotChannel, BotProperties, MsTeamsChannel, MsTeamsChannelProperties, Sku
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.resource.deployments import DeploymentsMgmtClient
from azure.mgmt.resource.deployments.models import Deployment, DeploymentProperties
from azure.mgmt.resource.resources import ResourceManagementClient
from azure.mgmt.resource.resources.models import ResourceGroup
from azure.mgmt.resource.subscriptions import SubscriptionClient
from dotenv import dotenv_values

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "private-agent/azure.yaml").is_file())
WORKDIR = ROOT / "private-agent"
cfg = dotenv_values(ROOT / ".env")

SUBSCRIPTION_ID = cfg["PRIVATE_DEMO_SUBSCRIPTION_ID"]
RESOURCE_GROUP = cfg["PRIVATE_DEMO_RESOURCE_GROUP"]
LOCATION = cfg["PRIVATE_DEMO_LOCATION"]
DEPLOYMENT_NAME = cfg["PRIVATE_DEMO_DEPLOYMENT_NAME"]
AZD_ENV = cfg["PRIVATE_DEMO_AZD_ENV"]
MODEL_NAME = cfg["PRIVATE_DEMO_MODEL_NAME"]
AGENT_NAME = "demo-hosted-agent"  # must match the service name in private-agent/azure.yaml
SAMPLE_COMMIT = "c7226f4392c07326f03dcaae0ec1e7bb93118783"
SAMPLE_PATH = "infrastructure/infrastructure-setup-bicep/11-private-network-basic-vnet"
SOURCE_DIR = WORKDIR / ".azure" / "notebook-state" / "foundry-samples"

def run(*args, cwd=ROOT):
    result = subprocess.run([str(arg) for arg in args], cwd=cwd, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(result.stderr[-3000:])
    return result.stdout.strip()

credential = AzureCliCredential(subscription=SUBSCRIPTION_ID)
TENANT_ID = SubscriptionClient(credential).subscriptions.get(SUBSCRIPTION_ID).tenant_id
PRINCIPAL_ID = cfg.get("PRIVATE_DEMO_PRINCIPAL_ID") or run("az", "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv")

resource_client = ResourceManagementClient(credential, SUBSCRIPTION_ID)
deployment_client = DeploymentsMgmtClient(credential, SUBSCRIPTION_ID)
authorization_client = AuthorizationManagementClient(credential, SUBSCRIPTION_ID)
cognitive_client = CognitiveServicesManagementClient(credential, SUBSCRIPTION_ID)
bot_client = AzureBotService(credential, SUBSCRIPTION_ID)
print("Target:", SUBSCRIPTION_ID, RESOURCE_GROUP, LOCATION)


Target: dcbc681e-a69d-4f95-bc8e-da6054697474 rg-foundry-private-teams-demo switzerlandnorth


## 2. Provision the private infrastructure (billable)

The first run clones the pinned sample, compiles `main.bicep` with `az bicep build`, and deploys it to the resource group with the Azure SDK. This takes roughly 15–30 minutes. Later runs (for example, after moving to a connected VM) only read the existing deployment's outputs. Set `REDEPLOY = True` to apply parameter changes.

The cell also assigns **Foundry User** on the new project to `PRINCIPAL_ID`, the tenant-member user object ID. Role propagation can take a few minutes. If the deployment fails, inspect its operations in the portal before retrying. A failed attempt can leave the agent subnet associated.


In [2]:
REDEPLOY = False

deployed = (resource_client.resource_groups.check_existence(RESOURCE_GROUP)
            and deployment_client.deployments.check_existence(RESOURCE_GROUP, DEPLOYMENT_NAME))
if REDEPLOY or not deployed:
    if not (SOURCE_DIR / ".git").exists():
        run("git", "init", "--quiet", SOURCE_DIR)
        run("git", "remote", "add", "origin", "https://github.com/microsoft-foundry/foundry-samples.git", cwd=SOURCE_DIR)
        run("git", "sparse-checkout", "set", SAMPLE_PATH, cwd=SOURCE_DIR)
    run("git", "fetch", "--depth=1", "--filter=blob:none", "origin", SAMPLE_COMMIT, cwd=SOURCE_DIR)
    run("git", "checkout", "--detach", "--quiet", SAMPLE_COMMIT, cwd=SOURCE_DIR)
    template = json.loads(run("az", "bicep", "build", "--file", SOURCE_DIR / SAMPLE_PATH / "main.bicep", "--stdout"))
    parameters = {key: {"value": value} for key, value in {
        "location": LOCATION,
        "aiServices": cfg["PRIVATE_DEMO_ACCOUNT_BASE"],
        "firstProjectName": cfg["PRIVATE_DEMO_PROJECT_BASE"],
        "vnetName": cfg["PRIVATE_DEMO_VNET_NAME"],
        "vnetAddressPrefix": cfg["PRIVATE_DEMO_VNET_PREFIX"],
        "agentSubnetPrefix": cfg["PRIVATE_DEMO_AGENT_SUBNET_PREFIX"],
        "peSubnetPrefix": cfg["PRIVATE_DEMO_PE_SUBNET_PREFIX"],
        "modelName": MODEL_NAME,
        "modelVersion": cfg["PRIVATE_DEMO_MODEL_VERSION"],
        "modelSkuName": cfg["PRIVATE_DEMO_MODEL_SKU"],
        "modelCapacity": int(cfg["PRIVATE_DEMO_MODEL_CAPACITY"]),
        "enableContainerRegistry": False,
    }.items()}
    resource_client.resource_groups.create_or_update(RESOURCE_GROUP, ResourceGroup(location=LOCATION))
    print("Deploying infrastructure...")
    deployment_client.deployments.begin_create_or_update(RESOURCE_GROUP, DEPLOYMENT_NAME, Deployment(
        properties=DeploymentProperties(mode="Incremental", template=template, parameters=parameters))).result()

deployment = deployment_client.deployments.get(RESOURCE_GROUP, DEPLOYMENT_NAME)
if deployment.properties.provisioning_state != "Succeeded":
    raise RuntimeError(f"Deployment is {deployment.properties.provisioning_state}; inspect it in the portal.")
outputs = deployment.properties.outputs
ACCOUNT_ID = outputs["accountId"]["value"]
ACCOUNT_NAME = outputs["accountName"]["value"]
PROJECT_NAME = outputs["projectName"]["value"]
PROJECT_ID = f"{ACCOUNT_ID}/projects/{PROJECT_NAME}"
PROJECT_ENDPOINT = f"https://{ACCOUNT_NAME}.services.ai.azure.com/api/projects/{PROJECT_NAME}"
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential, allow_preview=True)
print("Private project:", PROJECT_ENDPOINT)

foundry_user = f"/subscriptions/{SUBSCRIPTION_ID}/providers/Microsoft.Authorization/roleDefinitions/53ca6127-db72-4b80-b1b0-d745d6d5456d"
try:
    authorization_client.role_assignments.create(PROJECT_ID, str(uuid4()), RoleAssignmentCreateParameters(
        role_definition_id=foundry_user, principal_id=PRINCIPAL_ID, principal_type="User"))
    print("Foundry User assigned.")
except HttpResponseError as error:
    if "RoleAssignmentExists" not in str(error):
        raise
    print("Foundry User already assigned.")


Deploying infrastructure...
Private project: https://foundryteams4cfr.services.ai.azure.com/api/projects/teams4cfr
Foundry User assigned.


## 3. Private-connectivity gate

**Stop here until the kernel has private connectivity** (VPN with private DNS, or a connected VM). Don't enable Foundry public access as a workaround. After a kernel restart, rerun cells 3 and 5. Cell 5 only reads the existing deployment.

The next cell confirms that public network access is disabled and that the project data-plane API is reachable over the private endpoint.


In [ ]:
account = cognitive_client.accounts.get(RESOURCE_GROUP, ACCOUNT_NAME)
print("Foundry public network access:", account.properties.public_network_access)
print("Private API reachable. Agents:", [agent.name for agent in project.agents.list()])


## 4. Deploy and test the existing hosted agent

The [companion azd manifest](private-agent/azure.yaml) deploys [the existing source](agent/src/demo-hosted-agent/main.py) by using `codeConfiguration` (Python 3.13, no Dockerfile/ACR). Its azd environment is separate from `agent/.azure`. Bicep already created the project, so **don't run `azd up` or `azd provision`**. Each `azd deploy` creates a new agent version.

The smoke test uses `project.get_openai_client(agent_name=...)`, which targets the agent's dedicated Responses endpoint over the private network.


In [ ]:
if not (WORKDIR / ".azure" / AZD_ENV).exists():
    run("azd", "env", "new", AZD_ENV, "--subscription", SUBSCRIPTION_ID, "--location", LOCATION, "--no-prompt", cwd=WORKDIR)
for key, value in {
    "AZURE_SUBSCRIPTION_ID": SUBSCRIPTION_ID, "AZURE_TENANT_ID": TENANT_ID,
    "AZURE_RESOURCE_GROUP": RESOURCE_GROUP, "AZURE_LOCATION": LOCATION,
    "FOUNDRY_PROJECT_ENDPOINT": PROJECT_ENDPOINT, "AZURE_AI_PROJECT_ENDPOINT": PROJECT_ENDPOINT,
    "AZURE_AI_PROJECT_ID": PROJECT_ID, "AZURE_AI_MODEL_DEPLOYMENT_NAME": MODEL_NAME,
}.items():
    run("azd", "env", "set", key, value, "--environment", AZD_ENV, "--no-prompt", cwd=WORKDIR)
print(run("azd", "deploy", AGENT_NAME, "--environment", AZD_ENV, "--no-prompt", cwd=WORKDIR)[-2000:])


In [ ]:
openai_client = project.get_openai_client(agent_name=AGENT_NAME)
response = openai_client.responses.create(input="Which deployment type should I use for overnight batch summarisation?")
print(response.output_text)


## 5. Publish to Teams: follow the doc's steps 1–4

### Steps 1–2: agent identity and Azure Bot Service

The bot uses the agent identity's **client ID** (not its principal ID) as a single-tenant `msaAppId`. Its endpoint is the agent's Activity Protocol route, and its public network access is disabled. You don't need a bot secret, APIM, or tunnel. Azure Bot Service reaches Foundry through the restricted public Activity route, not the private endpoint.


In [ ]:
agent = project.agents.get(AGENT_NAME)
AGENT_CLIENT_ID = agent.instance_identity.client_id
print("Agent identity client ID:", AGENT_CLIENT_ID, "| Tenant:", TENANT_ID)

BOT_NAME = f"bot-{ACCOUNT_NAME}"
bot = bot_client.bots.create(RESOURCE_GROUP, BOT_NAME, Bot(
    location="global", kind="azurebot", sku=Sku(name="F0"),
    properties=BotProperties(
        display_name="Private Foundry demo",
        endpoint=f"{PROJECT_ENDPOINT}/agents/{AGENT_NAME}/endpoint/protocols/activityProtocol?api-version=2025-05-15-preview",
        msa_app_id=AGENT_CLIENT_ID, msa_app_tenant_id=TENANT_ID,
        msa_app_type="SingleTenant", public_network_access="Disabled")))
bot_client.channels.create(RESOURCE_GROUP, BOT_NAME, "MsTeamsChannel", BotChannel(
    location="global", properties=MsTeamsChannel(properties=MsTeamsChannelProperties(is_enabled=True))))
print("Bot with Teams channel:", bot.id)


### Step 3: enable the source-IP-filtered Activity route and Bot Service authorization

`enable_m365_public_endpoint=True` opens only the Activity Protocol route to Azure Bot Service / Microsoft 365 source IPs. Responses and the other project APIs stay private. The update **replaces** the protocol and authorization collections, so it keeps `responses` + `Entra` next to `activity` + `BotServiceRbac`. `BotServiceRbac` matches the `Shared` scope used in step 4.


In [ ]:
updated = project.agents.update_details(AGENT_NAME, agent_endpoint=AgentEndpointConfig(
    protocol_configuration=ProtocolConfiguration(
        responses=ResponsesProtocolConfiguration(),
        activity=ActivityProtocolConfiguration(enable_m365_public_endpoint=True)),
    authorization_schemes=[EntraAuthorizationScheme(), BotServiceRbacAuthorizationScheme()]))
print(json.dumps(updated.agent_endpoint.as_dict(), indent=2))


### Step 4: publish to Microsoft 365

`Shared` makes the agent visible only to you, under **Your agents**, without admin approval. Chat participants still need Foundry permissions on the project. `Tenant` would require Microsoft 365 admin approval.

Replace the developer metadata with your own values and don't put secrets in any field. Republishing the same `app_version` fails, so increment it only when you change the metadata. A successful call returns a `title_id`.


In [ ]:
published = project.agents.publish_to_microsoft365(
    AGENT_NAME,
    publish_scope="Shared",
    bot_service_arm_id=bot.id,
    publish_as_autopilot=False,
    app_version="1.0.0",
    agent_display_name="Private Foundry assistant",
    short_description="Deployment guidance from a private-networked Foundry hosted agent.",
    full_description="Authenticated Teams access to a private Foundry hosted agent without APIM.",
    developer_name="Contoso",
    developer_website_url="https://www.contoso.com",
    privacy_url="https://www.contoso.com/privacy",
    terms_of_use_url="https://www.contoso.com/terms",
)
print("Published title ID:", published.title_id)


## 6. Prove the scenario

1. In Teams, open the agent store with the same tenant-member account and find the agent under **Your agents**. The catalog cache can take up to about an hour; sign out and back in to refresh it rather than republishing.
2. Ask **Which deployment type should I use for overnight batch summarisation?** A reply confirms end-to-end channel delivery.
3. From a machine **outside** the private network, a Responses call to the same agent should fail with `403 NetworkAccessDenied`. From the connected kernel, the smoke test in cell 10 still succeeds.

A private call succeeding, an outside call being denied, and Teams working together demonstrate the design.

Troubleshooting: if Teams gets `403 NetworkAccessDenied`, check `enable_m365_public_endpoint`. For authorization errors, check `BotServiceRbac`, tenant membership, and the user's Foundry role. If a conversation gets stuck, send `/foundry_new_preview` to reset it.

## 7. Cleanup (optional)

Remove the app from Teams first. Deleting Azure resources doesn't clean up the Microsoft 365 catalog. The cell below deletes the whole dedicated resource group. If a capability host blocks subnet reuse, follow the sample's cleanup guidance.


In [ ]:
DELETE_RESOURCE_GROUP = False
if DELETE_RESOURCE_GROUP:
    resource_client.resource_groups.begin_delete(RESOURCE_GROUP)
    print("Deletion of", RESOURCE_GROUP, "started.")
